# Notebook 4: Patrones de Flujo - Generators, Streaming y Async

## 🔄 Recapitulación

| Notebook | Aprendiste | Ejemplo clave |
|----------|------------|---------------|
| **N1** | Datos y estructuras | `state = {"messages": [], "step": "init"}` |
| **N2** | POO, tipos, decoradores | `@tool`, `TypedDict`, `BaseModel` |
| **N3** | Errores, config, callbacks | `try/except`, `os.environ`, `on_llm_end()` |

## 🎯 Objetivo de este notebook

Aprender cómo **fluyen los datos** en programas reales:
- Procesar datos uno a uno (generators)
- Ver respuestas mientras se generan (streaming)
- Manejar recursos automáticamente (context managers)
- Hacer varias cosas "a la vez" (async/await)

## 🧠 ¿Por qué estos patrones?

Cuando usas un LLM:
- La respuesta llega **palabra por palabra** → Streaming
- Quieres ver la respuesta **mientras se genera** → Generators
- Necesitas **contar tokens/costos** → Context managers
- Quieres hacer **múltiples llamadas eficientemente** → Async

## 📋 Contenido

1. **Iteradores** → Fundamento (repaso y profundización)
2. **Generators** → `yield` y evaluación perezosa
3. **Streaming** → Procesar respuestas en tiempo real
4. **Context Managers** → El patrón `with`
5. **Async/Await** → Concurrencia sin hilos

## ⚠️ Nota importante

Este notebook cubre conceptos **más avanzados**. Si algo no queda claro:
1. Ejecuta el código paso a paso
2. Modifica los valores y observa qué cambia
3. No te preocupes si async/await es confuso al principio - lo dominarás con práctica

---
# PARTE 1: ITERADORES (Fundamento)

## Repaso: El bucle `for`

En el Notebook 1 usaste bucles `for`:

In [ ]:
# Repaso: for loop básico

frutas = ["manzana", "banana", "cereza"]

for fruta in frutas:
    print(f"Fruta: {fruta}")

## ¿Qué pasa realmente en un `for`?

Cuando escribes `for x in algo`, Python hace esto internamente:

1. Llama a `iter(algo)` para obtener un **iterador**
2. Llama a `next(iterador)` repetidamente para obtener cada elemento
3. Cuando no hay más elementos, `next()` lanza `StopIteration`

Veámoslo manualmente:

In [ ]:
# Viendo el for "por dentro"

frutas = ["manzana", "banana", "cereza"]

# Paso 1: Obtener el iterador
iterador = iter(frutas)
print(f"Tipo del iterador: {type(iterador)}")

# Paso 2: Obtener elementos uno a uno con next()
print(f"\nPrimer elemento: {next(iterador)}")
print(f"Segundo elemento: {next(iterador)}")
print(f"Tercer elemento: {next(iterador)}")

# Paso 3: Si pedimos otro, lanza StopIteration
try:
    print(next(iterador))
except StopIteration:
    print("\n¡No hay más elementos! (StopIteration)")

## ¿Por qué importa esto?

Entender iteradores es la base para entender **generators** y **streaming**.

La idea clave es:
> **No necesitas tener todos los elementos en memoria. Puedes producirlos uno a uno.**

Esto es crucial para:
- Respuestas de LLM que llegan palabra por palabra
- Procesar archivos enormes línea por línea
- Streams de datos en tiempo real

In [ ]:
# EJEMPLO: Diferencia entre lista e iterador

# Una lista tiene TODOS los elementos en memoria
lista = [1, 2, 3, 4, 5]
print(f"Lista: {lista}")
print(f"Longitud: {len(lista)}")
print(f"Puedo acceder al índice 2: {lista[2]}")

print()

# Un iterador produce elementos UNO A UNO
iterador = iter(lista)
print(f"Iterador: {iterador}")
# print(len(iterador))  # ❌ Error: no tiene longitud
# print(iterador[2])    # ❌ Error: no es indexable
print(f"Solo puedo obtener el siguiente: {next(iterador)}")

---
# PARTE 2: GENERATORS Y `yield`

## El problema: Generar muchos datos

Imagina que quieres generar los primeros 1,000,000 de números:

In [ ]:
# ❌ FORMA INEFICIENTE: Crear toda la lista en memoria

def obtener_numeros_lista(n):
    """Crea una lista con todos los números."""
    resultado = []
    for i in range(n):
        resultado.append(i)
    return resultado  # Devuelve TODO de una vez

# Para números pequeños, funciona bien
numeros = obtener_numeros_lista(10)
print(f"Lista pequeña: {numeros}")

# Pero para 1,000,000... usa mucha memoria
# numeros_grandes = obtener_numeros_lista(1_000_000)  # ¡Millones en memoria!

## La solución: `yield` (generators)

Un **generator** es una función que usa `yield` en lugar de `return`.

La diferencia clave:
- `return`: Devuelve un valor y **termina** la función
- `yield`: Devuelve un valor y **pausa** la función (se puede continuar)

In [ ]:
# ✅ FORMA EFICIENTE: Generator con yield

def obtener_numeros_generator(n):
    """Genera números uno a uno (no los guarda en memoria)."""
    for i in range(n):
        yield i  # Produce un valor y PAUSA

# Crear el generator
gen = obtener_numeros_generator(10)
print(f"¿Qué es gen? {gen}")
print(f"Tipo: {type(gen)}")

# Obtener valores uno a uno
print(f"\nPrimer valor: {next(gen)}")
print(f"Segundo valor: {next(gen)}")
print(f"Tercer valor: {next(gen)}")

In [ ]:
# Usar generator en un for loop (lo más común)

def obtener_numeros_generator(n):
    for i in range(n):
        yield i

print("Usando generator en for loop:")
for numero in obtener_numeros_generator(5):
    print(f"  Recibí: {numero}")

## Visualizando `yield` paso a paso

Veamos exactamente qué pasa cuando se ejecuta un generator:

In [ ]:
# Visualizar el flujo de un generator

def generator_con_mensajes():
    print("  [Generator] Iniciando...")
    
    print("  [Generator] Preparando valor 1...")
    yield "primero"
    print("  [Generator] Continuando después del primer yield...")
    
    print("  [Generator] Preparando valor 2...")
    yield "segundo"
    print("  [Generator] Continuando después del segundo yield...")
    
    print("  [Generator] Preparando valor 3...")
    yield "tercero"
    print("  [Generator] Finalizando...")


print("=== CREANDO EL GENERATOR ===")
gen = generator_con_mensajes()
print("(Nota: aún no se ejecutó nada del código interno)\n")

print("=== PRIMER next() ===")
valor1 = next(gen)
print(f"Recibí: '{valor1}'\n")

print("=== SEGUNDO next() ===")
valor2 = next(gen)
print(f"Recibí: '{valor2}'\n")

print("=== TERCER next() ===")
valor3 = next(gen)
print(f"Recibí: '{valor3}'")

## `yield` vs `return`: La diferencia clave

| Aspecto | `return` | `yield` |
|---------|----------|--------|
| Qué devuelve | Un valor final | Múltiples valores, uno a uno |
| La función... | Termina | Se pausa y puede continuar |
| Memoria | Todo en memoria | Un valor a la vez |
| Tipo de resultado | El valor directamente | Un objeto generator |

In [ ]:
# Comparación directa: return vs yield

def con_return():
    print("Ejecutando con return...")
    return [1, 2, 3]  # Devuelve todo y termina

def con_yield():
    print("Ejecutando con yield...")
    yield 1  # Devuelve 1 y pausa
    print("Continuando...")
    yield 2  # Devuelve 2 y pausa
    print("Casi terminando...")
    yield 3  # Devuelve 3 y pausa
    print("Fin")


print("=== CON RETURN ===")
resultado_return = con_return()
print(f"Resultado: {resultado_return}")
print(f"Tipo: {type(resultado_return)}")

print("\n=== CON YIELD ===")
resultado_yield = con_yield()  # ¡No se ejecuta aún!
print(f"Resultado: {resultado_yield}")
print(f"Tipo: {type(resultado_yield)}")

print("\nAhora iteramos:")
for valor in resultado_yield:
    print(f"  Valor: {valor}")

---
# PARTE 3: STREAMING (Aplicación Práctica)

## ¿Qué es streaming?

**Streaming** es recibir datos poco a poco, en lugar de esperar a que todo esté listo.

Piensa en:
- **Sin streaming**: Esperas 10 segundos y recibes toda la respuesta
- **Con streaming**: Empiezas a ver la respuesta inmediatamente, palabra por palabra

## ¿Por qué es importante para LLMs?

Los LLMs generan texto **token por token**. Con streaming:
- El usuario ve la respuesta mientras se genera
- Mejor experiencia de usuario
- Puedes interrumpir si la respuesta va mal
- Especialmente importante con **modelos locales** (más lentos)

In [ ]:
# SIMULACIÓN: LLM sin streaming (esperar todo)

import time

def llm_sin_streaming(prompt):
    """
    Simula un LLM que devuelve todo al final.
    El usuario debe esperar a que termine.
    """
    respuesta_completa = "Hola, soy un asistente virtual y estoy aquí para ayudarte."
    
    # Simular tiempo de procesamiento
    print("Procesando", end="")
    for _ in range(len(respuesta_completa.split())):
        time.sleep(0.2)
        print(".", end="", flush=True)
    print()
    
    return respuesta_completa


print("=== SIN STREAMING ===")
print("(Debes esperar a que termine...)\n")

respuesta = llm_sin_streaming("Hola")
print(f"\nRespuesta: {respuesta}")

In [ ]:
# SIMULACIÓN: LLM con streaming (ver palabra por palabra)

import time

def llm_con_streaming(prompt):
    """
    Simula un LLM que usa streaming.
    Devuelve palabras una a una con yield.
    """
    respuesta = "Hola, soy un asistente virtual y estoy aquí para ayudarte."
    palabras = respuesta.split()
    
    for palabra in palabras:
        time.sleep(0.2)  # Simular tiempo de generación
        yield palabra + " "


print("=== CON STREAMING ===")
print("(Ves la respuesta mientras se genera...)\n")

print("Respuesta: ", end="")
for chunk in llm_con_streaming("Hola"):
    print(chunk, end="", flush=True)  # flush=True fuerza la impresión inmediata

print("\n\n¡Terminado!")

## Simulación más realista: Chunks de texto

En LangChain, el streaming devuelve "chunks" (fragmentos) que tienen estructura:

In [ ]:
# SIMULACIÓN: Streaming con chunks estructurados (como LangChain)

import time
from dataclasses import dataclass

@dataclass
class AIMessageChunk:
    """Simula AIMessageChunk de LangChain."""
    content: str
    
    def __repr__(self):
        return f"AIMessageChunk(content='{self.content}')"


def llm_stream_chunks(prompt):
    """
    Simula el stream de LangChain.
    Devuelve objetos AIMessageChunk.
    """
    respuesta = "Hola, soy un asistente virtual y estoy aquí para ayudarte."
    
    for palabra in respuesta.split():
        time.sleep(0.15)
        yield AIMessageChunk(content=palabra + " ")


print("=== STREAMING CON CHUNKS ===")
print("Mostrando cada chunk:\n")

respuesta_completa = ""
for chunk in llm_stream_chunks("Hola"):
    print(f"  Chunk recibido: {chunk}")
    respuesta_completa += chunk.content

print(f"\nRespuesta completa: '{respuesta_completa.strip()}'")

## 🔗 Conexión con LangChain

En LangChain, el streaming funciona **exactamente así**:

```python
# Código real de LangChain (referencia)
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3")

# Streaming: recibir chunks uno a uno
for chunk in llm.stream("Explícame qué es Python"):
    print(chunk.content, end="", flush=True)
```

**Nota**: `.stream()` devuelve un generator, por eso puedes usarlo en un `for`.

In [ ]:
# EJEMPLO: Procesar streaming y contar tokens

import time

def llm_stream(prompt):
    """Simula streaming de un LLM."""
    respuesta = "Python es un lenguaje de programación versátil y fácil de aprender."
    for palabra in respuesta.split():
        time.sleep(0.1)
        yield palabra + " "


# Procesar el stream con estadísticas
print("=== STREAMING CON ESTADÍSTICAS ===")
print("Respuesta: ", end="")

tokens_count = 0
respuesta_completa = ""

for token in llm_stream("¿Qué es Python?"):
    print(token, end="", flush=True)
    tokens_count += 1
    respuesta_completa += token

print(f"\n\n📊 Estadísticas:")
print(f"   Tokens generados: {tokens_count}")
print(f"   Caracteres totales: {len(respuesta_completa)}")

---
# PARTE 4: CONTEXT MANAGERS (`with`)

## El problema: Recursos que necesitan limpieza

Algunos recursos necesitan ser "cerrados" o "limpiados" después de usarlos:
- Archivos abiertos
- Conexiones de red
- Contadores de tokens/costos

Si olvidas cerrarlos, pueden ocurrir problemas.

In [ ]:
# Ejemplo: Abrir y cerrar archivos manualmente

# ❌ FORMA MANUAL (propensa a errores)
archivo = open(".env.ejemplo", "r")
contenido = archivo.read()
archivo.close()  # ¡Fácil de olvidar!

print("Contenido del archivo:")
print(contenido[:100] + "...")

In [ ]:
# ✅ FORMA CON CONTEXT MANAGER (with)

with open(".env.ejemplo", "r") as archivo:
    contenido = archivo.read()
    # El archivo se cierra AUTOMÁTICAMENTE al salir del with

print("Contenido del archivo:")
print(contenido[:100] + "...")
print("\n✅ Archivo cerrado automáticamente")

## ¿Cómo funciona `with`?

El statement `with` usa dos métodos especiales:
- `__enter__()`: Se ejecuta al **entrar** al bloque with
- `__exit__()`: Se ejecuta al **salir** del bloque with (¡siempre!)

Veámoslo:

In [ ]:
# Crear nuestro propio context manager

class MiContextManager:
    """Un context manager simple para entender cómo funciona."""
    
    def __init__(self, nombre):
        self.nombre = nombre
        print(f"  [__init__] Creando {nombre}")
    
    def __enter__(self):
        print(f"  [__enter__] Entrando al contexto de {self.nombre}")
        return self  # Lo que devuelve __enter__ es lo que va después de 'as'
    
    def __exit__(self, tipo_exc, valor_exc, traceback):
        print(f"  [__exit__] Saliendo del contexto de {self.nombre}")
        # Si hubo error, los parámetros tendrán información
        if tipo_exc:
            print(f"  [__exit__] ¡Hubo un error!: {valor_exc}")
        return False  # False = no suprimir errores


print("=== USANDO EL CONTEXT MANAGER ===")

with MiContextManager("recurso_ejemplo") as cm:
    print(f"  [Dentro] Usando {cm.nombre}")
    print(f"  [Dentro] Haciendo trabajo...")

print("\n[Fuera] El contexto terminó")

In [ ]:
# __exit__ se ejecuta INCLUSO SI HAY ERROR

print("=== CONTEXT MANAGER CON ERROR ===")

try:
    with MiContextManager("recurso_con_error") as cm:
        print(f"  [Dentro] Usando {cm.nombre}")
        raise ValueError("¡Algo salió mal!")
        print("  [Dentro] Esta línea nunca se ejecuta")
except ValueError as e:
    print(f"\n[Fuera] Capturamos el error: {e}")

print("\n✅ El __exit__ se ejecutó a pesar del error")

## Forma simplificada: `contextlib`

Crear clases con `__enter__` y `__exit__` es verboso. Python ofrece una forma más simple:

In [ ]:
# Crear context managers con @contextmanager

from contextlib import contextmanager

@contextmanager
def temporizador(nombre):
    """
    Context manager que mide cuánto tiempo tarda el código.
    """
    import time
    
    print(f"⏱️ Iniciando '{nombre}'...")
    inicio = time.time()
    
    yield  # <-- Aquí se ejecuta el código dentro del 'with'
    
    fin = time.time()
    print(f"⏱️ '{nombre}' tardó {fin - inicio:.2f} segundos")


# Usar el context manager
import time

with temporizador("mi_operación"):
    print("  Haciendo trabajo...")
    time.sleep(1)
    print("  Trabajo completado")

## Ejemplo práctico: Contador de tokens

In [ ]:
# SIMULACIÓN: Context manager para contar tokens (como LangChain)

from contextlib import contextmanager


class TokenCounter:
    """Simula un contador de tokens como en LangChain."""
    
    def __init__(self):
        self.prompt_tokens = 0
        self.completion_tokens = 0
    
    @property
    def total_tokens(self):
        return self.prompt_tokens + self.completion_tokens
    
    @property
    def total_cost(self):
        # Precio simulado: $0.001 por 1000 tokens
        return self.total_tokens * 0.001 / 1000


@contextmanager
def track_tokens():
    """
    Context manager que rastrea tokens usados.
    Similar a get_openai_callback() de LangChain.
    """
    counter = TokenCounter()
    yield counter
    print(f"\n📊 Uso de tokens:")
    print(f"   Prompt: {counter.prompt_tokens}")
    print(f"   Completion: {counter.completion_tokens}")
    print(f"   Total: {counter.total_tokens}")
    print(f"   Costo estimado: ${counter.total_cost:.6f}")


def simular_llamada_llm(prompt, counter):
    """Simula una llamada a LLM actualizando el contador."""
    counter.prompt_tokens += len(prompt.split()) * 2  # Simulación
    
    respuesta = f"Respuesta simulada para: {prompt[:20]}..."
    counter.completion_tokens += len(respuesta.split()) * 2
    
    return respuesta


# Usar el context manager
print("=== RASTREO DE TOKENS ===")

with track_tokens() as counter:
    resp1 = simular_llamada_llm("¿Cuál es la capital de Francia?", counter)
    print(f"Respuesta 1: {resp1}")
    
    resp2 = simular_llamada_llm("Explícame la fotosíntesis en detalle por favor", counter)
    print(f"Respuesta 2: {resp2}")

## 🔗 Conexión con LangChain

LangChain usa context managers para rastrear tokens y costos:

```python
# Código real de LangChain (referencia)
from langchain_community.callbacks import get_openai_callback
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

with get_openai_callback() as cb:
    response1 = llm.invoke("Hola")
    response2 = llm.invoke("¿Cómo estás?")
    
    print(f"Tokens usados: {cb.total_tokens}")
    print(f"Costo: ${cb.total_cost:.4f}")
```

---
# PARTE 5: ASYNC/AWAIT

## El problema: Esperar bloquea todo

Imagina que tienes que hacer 3 llamadas a un LLM, y cada una tarda 2 segundos:

In [ ]:
# CÓDIGO SÍNCRONO: Una cosa a la vez

import time

def llamar_llm_sync(pregunta):
    """Simula una llamada a LLM que tarda 1 segundo."""
    print(f"  Procesando: '{pregunta[:30]}...'")
    time.sleep(1)  # Simular tiempo de respuesta
    return f"Respuesta a: {pregunta[:20]}"


print("=== LLAMADAS SÍNCRONAS (secuenciales) ===")
inicio = time.time()

preguntas = [
    "¿Cuál es la capital de Francia?",
    "¿Cuántos planetas hay en el sistema solar?",
    "¿Quién escribió Don Quijote?"
]

for pregunta in preguntas:
    respuesta = llamar_llm_sync(pregunta)
    print(f"  ✅ {respuesta}")

print(f"\n⏱️ Tiempo total: {time.time() - inicio:.1f} segundos")
print("(3 llamadas × 1 segundo = 3 segundos)")

## La metáfora del restaurante 🍽️

Imagina que eres un mesero en un restaurante:

### Forma síncrona (bloqueante):
1. Tomas el pedido de la mesa 1
2. **Esperas en la cocina** hasta que esté listo
3. Llevas el plato a la mesa 1
4. Recién ahora puedes atender a la mesa 2

### Forma asíncrona (no bloqueante):
1. Tomas el pedido de la mesa 1 → lo mandas a la cocina
2. **Mientras cocina**, tomas el pedido de la mesa 2 → lo mandas
3. Tomas el pedido de la mesa 3 → lo mandas
4. Cuando cada plato está listo, lo llevas

**Mismo tiempo de cocina, pero atiendes a todos más rápido.**

## Conceptos clave de async

| Concepto | Significado |
|----------|-------------|
| `async def` | Define una función asíncrona (coroutine) |
| `await` | "Espera aquí, pero deja que otros trabajen mientras" |
| `asyncio` | Librería para ejecutar código asíncrono |
| Event Loop | El "coordinador" que maneja las tareas |

In [ ]:
# ASYNC BÁSICO: Tu primera función asíncrona

import asyncio

# Definir una función asíncrona con 'async def'
async def saludar(nombre):
    """Función asíncrona simple."""
    print(f"Hola, {nombre}!")
    
    # await asyncio.sleep() es la versión async de time.sleep()
    await asyncio.sleep(1)
    
    print(f"Adiós, {nombre}!")
    return f"Saludé a {nombre}"


# Para ejecutar una función async, necesitamos await
# En Jupyter, podemos usar await directamente
resultado = await saludar("María")
print(f"Resultado: {resultado}")

In [ ]:
# DIFERENCIA: time.sleep vs asyncio.sleep

import asyncio
import time

async def tarea_con_await(nombre, segundos):
    print(f"  [{nombre}] Iniciando...")
    await asyncio.sleep(segundos)  # ← No bloquea, permite otras tareas
    print(f"  [{nombre}] ¡Terminé después de {segundos}s!")
    return nombre


print("=== EJECUTANDO TAREAS SECUENCIALMENTE ===")
inicio = time.time()

# Aunque son async, si las esperamos una por una, son secuenciales
await tarea_con_await("Tarea A", 1)
await tarea_con_await("Tarea B", 1)

print(f"\n⏱️ Tiempo: {time.time() - inicio:.1f}s (secuencial = 2s)")

In [ ]:
# EJECUTAR TAREAS EN PARALELO con asyncio.gather

import asyncio
import time

async def tarea_con_await(nombre, segundos):
    print(f"  [{nombre}] Iniciando...")
    await asyncio.sleep(segundos)
    print(f"  [{nombre}] ¡Terminé después de {segundos}s!")
    return nombre


print("=== EJECUTANDO TAREAS EN PARALELO ===")
inicio = time.time()

# asyncio.gather ejecuta todas las tareas "a la vez"
resultados = await asyncio.gather(
    tarea_con_await("Tarea A", 1),
    tarea_con_await("Tarea B", 1),
    tarea_con_await("Tarea C", 1),
)

print(f"\n⏱️ Tiempo: {time.time() - inicio:.1f}s (paralelo ≈ 1s)")
print(f"Resultados: {resultados}")

## Aplicación: Múltiples llamadas a LLM en paralelo

In [ ]:
# SIMULACIÓN: Llamadas async a un LLM

import asyncio
import time

async def llamar_llm_async(pregunta):
    """
    Simula una llamada asíncrona a un LLM.
    Similar a: await llm.ainvoke(pregunta)
    """
    print(f"  📤 Enviando: '{pregunta[:25]}...'")
    await asyncio.sleep(1)  # Simular tiempo de respuesta
    respuesta = f"Respuesta a: {pregunta[:15]}..."
    print(f"  📥 Recibida respuesta para: '{pregunta[:15]}...'")
    return respuesta


preguntas = [
    "¿Cuál es la capital de Francia?",
    "¿Cuántos planetas hay en el sistema solar?",
    "¿Quién escribió Don Quijote?",
    "¿Cuál es el río más largo del mundo?",
]

print("=== LLAMADAS ASYNC (en paralelo) ===")
inicio = time.time()

# Crear tareas para todas las preguntas
tareas = [llamar_llm_async(p) for p in preguntas]

# Ejecutar todas en paralelo
respuestas = await asyncio.gather(*tareas)

print(f"\n⏱️ Tiempo total: {time.time() - inicio:.1f}s")
print(f"(4 llamadas en paralelo ≈ 1 segundo, no 4)")

print("\n📋 Respuestas:")
for r in respuestas:
    print(f"  - {r}")

## ¿Cuándo usar async?

| Situación | ¿Usar async? |
|-----------|-------------|
| Una sola llamada a LLM | No necesario |
| Múltiples llamadas a LLM | ✅ Sí, para paralelizar |
| Aplicación web/API | ✅ Sí, para manejar múltiples usuarios |
| Script simple | No necesario |
| Procesamiento de CPU intensivo | ❌ No ayuda (usar multiprocessing) |

**Regla práctica**: Async ayuda cuando estás **esperando** algo (red, disco, API). No ayuda cuando estás **calculando** algo.

## 🔗 Conexión con LangChain

LangChain ofrece versiones async de sus métodos:

```python
# Código real de LangChain (referencia)
from langchain_ollama import ChatOllama
import asyncio

llm = ChatOllama(model="llama3")

# Versión síncrona
response = llm.invoke("Hola")  # Bloquea hasta terminar

# Versión asíncrona
response = await llm.ainvoke("Hola")  # Permite otras tareas

# Múltiples llamadas en paralelo
responses = await asyncio.gather(
    llm.ainvoke("Pregunta 1"),
    llm.ainvoke("Pregunta 2"),
    llm.ainvoke("Pregunta 3"),
)
```

**Métodos async en LangChain**:
- `.ainvoke()` → versión async de `.invoke()`
- `.astream()` → versión async de `.stream()`
- `.abatch()` → versión async de `.batch()`

---
# INTEGRACIÓN: Streaming asíncrono

Combinemos generators (streaming) con async:

In [ ]:
# ASYNC GENERATOR: Streaming asíncrono

import asyncio

async def llm_astream(prompt):
    """
    Simula streaming asíncrono.
    Similar a: async for chunk in llm.astream(prompt)
    """
    respuesta = f"Hola! Procesando tu pregunta sobre '{prompt[:20]}'. Aquí está mi respuesta detallada."
    
    for palabra in respuesta.split():
        await asyncio.sleep(0.1)  # Simular tiempo de generación
        yield palabra + " "


print("=== STREAMING ASÍNCRONO ===")
print("Respuesta: ", end="")

# async for para iterar sobre un async generator
async for chunk in llm_astream("la historia de Python"):
    print(chunk, end="", flush=True)

print("\n\n✅ Streaming completado")

In [ ]:
# EJEMPLO COMPLETO: Simulador de agente con streaming async

import asyncio
from contextlib import asynccontextmanager
from dataclasses import dataclass, field
from typing import AsyncGenerator


@dataclass
class AgentMetrics:
    """Métricas del agente."""
    tokens_generated: int = 0
    chunks_count: int = 0


@asynccontextmanager
async def track_agent_metrics():
    """Context manager async para rastrear métricas."""
    metrics = AgentMetrics()
    try:
        yield metrics
    finally:
        print(f"\n📊 Métricas finales:")
        print(f"   Chunks: {metrics.chunks_count}")
        print(f"   Tokens: {metrics.tokens_generated}")


async def agent_stream(prompt: str, metrics: AgentMetrics) -> AsyncGenerator[str, None]:
    """Agente simulado con streaming."""
    respuestas = {
        "clima": "El clima hoy es soleado con temperatura de 22 grados centígrados.",
        "hora": "La hora actual es las 14:30 de la tarde.",
        "default": "Soy un asistente y estoy aquí para ayudarte con lo que necesites."
    }
    
    # Seleccionar respuesta
    if "clima" in prompt.lower():
        respuesta = respuestas["clima"]
    elif "hora" in prompt.lower():
        respuesta = respuestas["hora"]
    else:
        respuesta = respuestas["default"]
    
    # Streaming palabra por palabra
    for palabra in respuesta.split():
        await asyncio.sleep(0.1)
        metrics.chunks_count += 1
        metrics.tokens_generated += len(palabra)
        yield palabra + " "


# Ejecutar el agente
print("=== AGENTE CON STREAMING ASYNC ===\n")

async with track_agent_metrics() as metrics:
    print("Usuario: ¿Cómo está el clima?")
    print("Agente: ", end="")
    
    async for chunk in agent_stream("¿Cómo está el clima?", metrics):
        print(chunk, end="", flush=True)

---
# 📝 Resumen del Notebook 4

## Lo que aprendiste

| Concepto | Para qué | Patrón LangChain |
|----------|----------|------------------|
| **Iteradores** | Entender cómo funciona `for` | Fundamento |
| **Generators (`yield`)** | Producir datos uno a uno | Base de streaming |
| **Streaming** | Ver respuestas en tiempo real | `for chunk in llm.stream()` |
| **Context Managers** | Manejar recursos automáticamente | `with get_openai_callback()` |
| **Async/Await** | Múltiples llamadas en paralelo | `await llm.ainvoke()` |

## Código real que ahora entiendes

```python
# Streaming
for chunk in llm.stream("Hola"):
    print(chunk.content, end="")

# Context manager para tracking
with get_openai_callback() as cb:
    response = llm.invoke("Hola")
    print(f"Tokens: {cb.total_tokens}")

# Async para paralelizar
responses = await asyncio.gather(
    llm.ainvoke("P1"),
    llm.ainvoke("P2"),
)

# Streaming async
async for chunk in llm.astream("Hola"):
    print(chunk.content, end="")
```

## Siguiente paso

**Notebook 5**: Primeros pasos con LangChain y modelos locales (Ollama)

¡Ahora tienes todos los fundamentos de Python para entender código de agentes!